# 4.3 앙상블 학습 개요

## 앙상블 학습(Ensemble Learning) 개념
여러 개의 분류기(Classifier)를 생성하고 그 예측을 결합하여 단일 분류기보다 더 정확한 최종 예측을 도출하는 기법.  
단일 모델의 약점을 다수의 모델들을 결합하여 보완하는 것이 핵심 아이디어이다.

## 앙상블 학습의 유형
- **Voting (보팅)**: 서로 다른 알고리즘의 분류기들을 결합. 같은 데이터셋을 사용
- **Bagging (배깅)**: 같은 알고리즘의 분류기들을 결합. 데이터 샘플링을 다르게(부트스트래핑) 가져감. 대표적으로 Random Forest
- **Boosting (부스팅)**: 여러 개의 약한 학습기를 순차적으로 학습. 이전 분류기가 틀린 예측에 가중치를 부여하며 학습. 대표적으로 GBM, XGBoost, LightGBM

## Voting의 두 가지 방식
- **Hard Voting**: 다수결 방식. 예측한 결과값들 중 다수의 분류기가 결정한 예측값을 최종 보팅 결괏값으로 선정
- **Soft Voting**: 분류기들의 레이블 값 결정 확률을 모두 평균내어 가장 확률이 높은 레이블 값을 최종 보팅 결괏값으로 선정. 일반적으로 Soft Voting이 예측 성능이 더 좋아 자주 사용됨

## Voting Classifier 실습
위스콘신 유방암 데이터셋을 활용해 로지스틱 회귀와 KNN을 Soft Voting으로 결합한다.

In [ ]:
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 위스콘신 유방암 데이터 로딩 (이진 분류 문제: 악성/양성)
cancer = load_breast_cancer()

# 피처 데이터를 DataFrame으로 변환하여 데이터 구조 확인
data_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
data_df.head(3)

In [ ]:
# 개별 모델 정의: 로지스틱 회귀와 KNN
# LogisticRegression의 solver는 liblinear로 지정 (소규모 데이터셋에 적합)
lr_clf = LogisticRegression(solver='liblinear')
knn_clf = KNeighborsClassifier(n_neighbors=8)

# VotingClassifier에 estimators로 (이름, 모델) 튜플의 리스트를 전달
# voting='soft'로 지정하여 확률 평균 기반의 Soft Voting 수행
vo_clf = VotingClassifier(estimators=[('LR', lr_clf), ('KNN', knn_clf)], voting='soft')

# 학습/테스트 데이터 분할 (8:2 비율, 재현성을 위해 random_state 고정)
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target,
                                                    test_size=0.2, random_state=156)

# Voting 분류기 학습, 예측, 평가
vo_clf.fit(X_train, y_train)
pred = vo_clf.predict(X_test)
print('Voting 분류기 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

# 비교를 위해 개별 모델의 성능도 확인
# Voting이 항상 개별 모델보다 성능이 좋은 것은 아니지만, 일반적으로 더 안정적인 성능을 보임
classifiers = [lr_clf, knn_clf]
for classifier in classifiers:
    classifier.fit(X_train, y_train)
    pred = classifier.predict(X_test)
    class_name = classifier.__class__.__name__
    print('{0} 정확도: {1:.4f}'.format(class_name, accuracy_score(y_test, pred)))

### 4.3 핵심 정리
- 앙상블은 여러 분류기를 결합해 단일 모델의 한계를 보완하는 기법이다.
- Voting은 서로 다른 알고리즘을, Bagging은 같은 알고리즘에 서로 다른 데이터 샘플을 활용한다.
- Soft Voting은 예측 확률의 평균을 사용하므로 Hard Voting보다 일반적으로 더 좋은 성능을 보인다.
- 단, Voting이 항상 개별 분류기보다 성능이 우수한 것은 아니라는 점에 유의해야 한다.

# 4.4 Random Forest

## Random Forest 개념
Bagging의 대표적인 알고리즘으로, 여러 개의 결정 트리(Decision Tree)를 학습시켜 그 결과를 결합하는 앙상블 모델이다.  
각 트리는 부트스트래핑(Bootstrapping)으로 추출된 서로 다른 데이터 샘플을 기반으로 학습되며,  
분류 시에는 다수결 투표, 회귀 시에는 평균값을 통해 최종 예측을 결정한다.

## Random Forest의 특징
- 결정 트리의 단점인 과적합(overfitting)을 줄여줌
- 각 트리가 학습할 때 전체 피처 중 일부만 무작위로 선택하여 분할 → 트리 간 다양성 확보
- 비교적 빠른 학습 속도와 안정적인 성능
- 하이퍼파라미터 튜닝이 비교적 직관적

## 주요 하이퍼파라미터
- `n_estimators`: 학습할 트리의 개수 (기본 100, 많을수록 성능 향상되나 속도 저하)
- `max_depth`: 트리의 최대 깊이 (과적합 방지)
- `min_samples_split`: 노드를 분할하기 위한 최소 샘플 수
- `min_samples_leaf`: 리프 노드가 되기 위한 최소 샘플 수
- `max_features`: 분할에 사용할 피처의 최대 개수 (기본값 'sqrt')

## 사용자 행동 인식(Human Activity Recognition) 데이터셋 로딩
스마트폰 센서 데이터로 사용자의 행동(걷기, 앉기 등)을 분류하는 다중 클래스 데이터셋.

In [ ]:
# features.txt 파일에 중복된 피처명이 존재하므로 이를 수정해주는 함수
# 중복되는 컬럼명 뒤에 _1, _2와 같이 일련번호를 붙여 고유한 컬럼명으로 변환
def get_new_feature_name_df(old_feature_name_df):
    # 같은 컬럼명별로 누적 개수를 계산 (중복 여부 판별용)
    feature_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(),
                                  columns=['dup_cnt'])
    feature_dup_df = feature_dup_df.reset_index()
    # 원본 컬럼명 DF와 중복 카운트 DF를 병합
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
    # 중복된 경우(dup_cnt > 0)에만 컬럼명 뒤에 _숫자를 붙임
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(
        lambda x: x[0] + '_' + str(x[1]) if x[1] > 0 else x[0], axis=1)
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

In [ ]:
import pandas as pd

# 사용자 행동 인식 데이터셋을 로딩하는 함수 정의
def get_human_dataset():

    # 피처명 파일을 공백 구분자 기준으로 로딩
    feature_name_df = pd.read_csv('./human_activity/features.txt', sep='\s+',
                                  header=None, names=['column_index', 'column_name'])

    # 중복된 피처명을 수정한 새로운 DataFrame 생성
    new_feature_name_df = get_new_feature_name_df(feature_name_df)

    # 컬럼명을 리스트 형태로 변환 (DataFrame의 columns 인자로 사용하기 위함)
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()

    # 학습용/테스트용 피처 데이터 로딩 (공백 구분자, 컬럼명은 위에서 만든 feature_name 사용)
    X_train = pd.read_csv('./human_activity/train/X_train.txt', sep='\s+', names=feature_name)
    X_test = pd.read_csv('./human_activity/test/X_test.txt', sep='\s+', names=feature_name)

    # 학습용/테스트용 레이블 데이터 로딩 (컬럼명은 'action'으로 지정)
    y_train = pd.read_csv('./human_activity/train/y_train.txt', sep='\s+', header=None, names=['action'])
    y_test = pd.read_csv('./human_activity/test/y_test.txt', sep='\s+', header=None, names=['action'])

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = get_human_dataset()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 데이터셋 재로딩
X_train, X_test, y_train, y_test = get_human_dataset()

# 기본 하이퍼파라미터로 랜덤 포레스트 학습
# random_state를 고정하여 재현 가능한 결과 확보
rf_clf = RandomForestClassifier(random_state=0)
rf_clf.fit(X_train, y_train)
pred = rf_clf.predict(X_test)
accuracy = accuracy_score(y_test, pred)
print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy))

## GridSearchCV를 활용한 하이퍼파라미터 튜닝
교차 검증을 통해 최적의 하이퍼파라미터 조합을 탐색한다.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 탐색할 하이퍼파라미터 조합 정의
# n_estimators는 100으로 고정 (튜닝 시간 단축 목적), 나머지 세 가지를 조합 탐색
params = {
    'n_estimators': [100],
    'max_depth': [6, 8, 10, 12],
    'min_samples_leaf': [8, 12, 18],
    'min_samples_split': [8, 16, 20]
}

# n_jobs=-1로 설정하여 모든 CPU 코어를 사용 (학습 속도 향상)
rf_clf = RandomForestClassifier(random_state=0, n_jobs=-1)
# cv=2로 2-Fold 교차 검증 수행 (조합 수가 많아 시간 절약 목적)
grid_cv = GridSearchCV(rf_clf, param_grid=params, cv=2, n_jobs=-1)
grid_cv.fit(X_train, y_train)

print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

In [ ]:
# 위에서 찾은 최적 하이퍼파라미터를 적용하되, n_estimators는 300으로 늘려 성능 향상 도모
# 일반적으로 트리 수를 늘리면 예측 안정성이 높아짐
rf_clf1 = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=8,
                                 min_samples_split=8, random_state=0)
rf_clf1.fit(X_train, y_train)
pred = rf_clf1.predict(X_test)
print('예측 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

## 피처 중요도(Feature Importance) 시각화
랜덤 포레스트는 각 피처가 분할 기준으로 얼마나 자주, 효과적으로 사용되었는지를 기반으로 중요도를 산출한다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# 학습된 모델의 feature_importances_ 속성을 통해 각 피처의 중요도 추출
ftr_importances_values = rf_clf1.feature_importances_
# Series로 변환하여 인덱스(피처명)와 함께 관리
ftr_importances = pd.Series(ftr_importances_values, index=X_train.columns)
# 중요도 내림차순 정렬 후 상위 20개만 추출
ftr_top20 = ftr_importances.sort_values(ascending=False)[:20]

# 상위 20개 피처를 가로 막대그래프로 시각화
plt.figure(figsize=(8, 6))
plt.title('Feature importances Top 20')
sns.barplot(x=ftr_top20, y=ftr_top20.index)
plt.show()

### 4.4 핵심 정리
- 랜덤 포레스트는 Bagging 기반 앙상블로, 부트스트래핑된 데이터에 독립적으로 학습된 결정 트리들의 결과를 결합한다.
- 각 트리가 무작위로 선택된 피처 일부만 사용해 분할하므로 트리 간 다양성이 확보되고, 결과적으로 과적합이 완화된다.
- 주요 하이퍼파라미터로는 n_estimators, max_depth, min_samples_split, min_samples_leaf 등이 있으며, GridSearchCV로 탐색할 수 있다.
- feature_importances_ 속성을 통해 어떤 피처가 모델의 예측에 큰 영향을 미쳤는지 해석할 수 있다.

# 6.2 PCA (Principal Component Analysis)

## PCA 개념
PCA(주성분 분석)는 가장 대표적인 차원 축소(Dimensionality Reduction) 기법으로,  
고차원 데이터의 분산을 최대한 보존하면서 더 낮은 차원의 공간으로 데이터를 투영하는 비지도 학습 방법이다.

## PCA의 핵심 원리
1. 데이터의 분산이 가장 큰 방향을 첫 번째 주성분(PC1)으로 설정
2. PC1에 직교하면서 그 다음으로 분산이 큰 방향을 두 번째 주성분(PC2)으로 설정
3. 이 과정을 반복하여 원하는 차원만큼 주성분을 추출
4. 각 주성분은 서로 독립적(직교)이며, 원본 피처들의 선형 결합으로 표현됨

## 수학적 관점
- 입력 데이터의 공분산 행렬(Covariance Matrix)에 대한 고유값 분해(Eigen Decomposition)를 수행
- 고유벡터(Eigenvector)가 주성분의 방향, 고유값(Eigenvalue)이 해당 방향의 분산 크기를 의미
- 고유값이 큰 순서대로 정렬한 고유벡터가 PC1, PC2, ... 순으로 채택됨

## PCA의 활용
- 차원 축소를 통한 시각화 (2D, 3D)
- 다중공선성(Multicollinearity) 문제 완화
- 노이즈 제거 및 학습 속도 향상

## 주의사항
- PCA 적용 전 반드시 **표준화(Standardization)** 가 필요함 → 피처의 스케일 차이가 분산에 영향을 주기 때문
- PCA는 분산이 큰 방향이 정보의 중요도와 일치한다고 가정하므로, 이 가정이 깨질 경우 성능 저하 가능

## 1) 붓꽃(Iris) 데이터셋에 PCA 적용
4개의 피처를 가진 붓꽃 데이터를 2개의 주성분으로 축소하고, 시각화 및 분류 성능을 비교한다.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# 사이킷런 내장 붓꽃 데이터셋 로딩
iris = load_iris()

# 넘파이 배열을 Pandas DataFrame으로 변환 (다루기 편한 형태로)
columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
irisDF = pd.DataFrame(iris.data, columns=columns)
irisDF['target'] = iris.target
irisDF.head(3)

In [ ]:
# 원본 피처 중 sepal_length, sepal_width 2개만 사용하여 시각화
# 품종별로 마커를 다르게 지정: setosa는 세모, versicolor는 네모, virginica는 동그라미
markers = ['^', 's', 'o']

# 각 target 값별로 데이터를 분리하여 산점도 그리기
for i, marker in enumerate(markers):
    x_axis_data = irisDF[irisDF['target'] == i]['sepal_length']
    y_axis_data = irisDF[irisDF['target'] == i]['sepal_width']
    plt.scatter(x_axis_data, y_axis_data, marker=marker, label=iris.target_names[i])

plt.legend()
plt.xlabel('sepal length')
plt.ylabel('sepal width')
plt.show()
# 결과 해석: setosa는 비교적 잘 구분되지만, versicolor와 virginica는 상당히 겹쳐 보임

In [ ]:
from sklearn.preprocessing import StandardScaler

# PCA 적용 전 반드시 표준화를 수행해야 함 (피처 간 스케일 차이가 분산 계산에 영향을 주기 때문)
# StandardScaler는 각 피처를 평균 0, 분산 1로 변환 (단, 데이터의 분포 형태 자체는 보존됨)
# target 컬럼은 제외하고 입력 피처만 변환
iris_scaled = StandardScaler().fit_transform(irisDF.iloc[:, :-1])

In [ ]:
from sklearn.decomposition import PCA

# 주성분 2개로 차원 축소 (4차원 -> 2차원)
pca = PCA(n_components=2)

# fit()으로 주성분 방향(고유벡터)을 학습한 후, transform()으로 실제 변환 수행
pca.fit(iris_scaled)
iris_pca = pca.transform(iris_scaled)
print(iris_pca.shape)  # (150, 2): 샘플 수는 그대로, 피처 차원만 2로 축소됨

In [ ]:
# PCA 변환된 데이터를 DataFrame으로 변환하여 시각화 준비
pca_columns = ['pca_component_1', 'pca_component_2']
irisDF_pca = pd.DataFrame(iris_pca, columns=pca_columns)
irisDF_pca['target'] = iris.target
irisDF_pca.head(3)

In [ ]:
# PCA 변환된 2개의 주성분을 축으로 산점도 시각화
markers = ['^', 's', 'o']

for i, marker in enumerate(markers):
    x_axis_data = irisDF_pca[irisDF_pca['target'] == i]['pca_component_1']
    y_axis_data = irisDF_pca[irisDF_pca['target'] == i]['pca_component_2']
    plt.scatter(x_axis_data, y_axis_data, marker=marker, label=iris.target_names[i])

plt.legend()
plt.xlabel('pca_component_1')
plt.ylabel('pca_component_2')
plt.show()
# 결과 해석: pca_component_1 축만으로도 세 품종이 비교적 잘 구분됨
# 즉, PC1이 데이터 분산의 대부분을 설명하고 있음을 시각적으로 확인 가능

In [ ]:
# 각 주성분이 전체 분산 중 차지하는 비율 확인
# explained_variance_ratio_는 각 PC가 설명하는 분산의 비율을 의미
print(pca.explained_variance_ratio_)
# 예: [0.7277, 0.2303]이라면 PC1이 약 72.8%, PC2가 약 23%의 분산을 설명
# 두 PC만으로 전체 분산의 약 95.8%를 설명 → 차원 축소에도 정보 손실이 적음

### 원본 데이터 vs PCA 변환 데이터의 분류 성능 비교

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# 원본 4개 피처 데이터로 랜덤 포레스트 학습 및 3-Fold 교차 검증
rcf = RandomForestClassifier(random_state=156)
scores = cross_val_score(rcf, iris.data, iris.target, scoring='accuracy', cv=3)
print('원본 데이터 교차 검증 개별 정확도:', scores)
print('원본 데이터 평균 정확도:', np.mean(scores))

In [ ]:
# PCA로 차원 축소된 2개 피처 데이터로 동일하게 교차 검증
pca_X = irisDF_pca[['pca_component_1', 'pca_component_2']]
scores_pca = cross_val_score(rcf, pca_X, iris.target, scoring='accuracy', cv=3)
print('PCA 변환 데이터 교차 검증 개별 정확도:', scores_pca)
print('PCA 변환 데이터 평균 정확도:', np.mean(scores_pca))
# 결과 해석: 피처 수가 절반으로 줄었음에도 성능 차이가 크지 않음
# 차원 축소를 통해 효율성을 얻으면서도 정보 손실을 최소화한 결과

## 2) Credit Card 데이터셋에 PCA 적용
23개의 피처를 가진 신용카드 연체 데이터셋에서 PCA를 통해 다중공선성을 가진 변수들을 효율적으로 압축한다.

In [ ]:
import pandas as pd

# header=1로 의미 없는 첫 번째 행을 건너뛰고 두 번째 행을 컬럼명으로 사용
# iloc[0:, 1:]로 기존 ID 컬럼 제거
df = pd.read_excel('pca_credit_card.xls', header=1, sheet_name='Data').iloc[0:, 1:]
print(df.shape)
df.head(3)

In [ ]:
# 컬럼명 정리: PAY_0를 PAY_1로 변경(일관성), 타겟 컬럼명 단축
df.rename(columns={'PAY_0': 'PAY_1', 'default payment next month': 'default'}, inplace=True)
# 타겟 변수(연체 여부)와 피처 분리
y_target = df['default']
X_features = df.drop('default', axis=1)

In [ ]:
# 피처 데이터의 정보 확인 (데이터 타입, 결측치 여부 등)
X_features.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# 피처 간 상관관계 분석 (히트맵 시각화)
# 상관계수가 높은 피처들은 다중공선성을 가질 가능성이 높음 → PCA로 압축 가능한 후보
corr = X_features.corr()
plt.figure(figsize=(14, 14))
sns.heatmap(corr, annot=True, fmt='.1g')
# 결과 해석: BILL_AMT1 ~ BILL_AMT6 (월별 청구액)이 서로 매우 높은 상관관계를 가짐
# 이는 PCA로 효과적으로 압축할 수 있는 좋은 후보군

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 상관관계가 높은 BILL_AMT, PAY, PAY_AMT 관련 컬럼들을 묶어서 PCA 적용 대상으로 선정
cols_bill = ['BILL_AMT' + str(i) for i in range(1, 7)]
cols_pay = ['PAY_' + str(i) for i in range(1, 7)]
cols_amt = ['PAY_AMT' + str(i) for i in range(1, 7)]
print(cols_bill)
cols_bill.extend(cols_pay)
cols_bill.extend(cols_amt)
print('대상 속성명:', cols_bill)

# 선정된 컬럼들에 대해 표준화 수행
scaler = StandardScaler()
df_cols_scaled = scaler.fit_transform(X_features[cols_bill])
X_features.loc[:, cols_bill] = df_cols_scaled

# 주성분 2개로 PCA 수행하여 설명 분산 비율 확인
pca = PCA(n_components=2)
pca.fit(df_cols_scaled)
print('PCA Component별 변동성:', pca.explained_variance_ratio_)
# 18개 피처를 2개 PC로 압축했음에도 상당한 분산이 보존됨 → 다중공선성 효과 확인

### 원본 23개 피처 vs PCA 6개 컴포넌트 분류 성능 비교

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 원본 23개 피처로 랜덤 포레스트 학습 및 3-Fold 교차 검증
rcf = RandomForestClassifier(n_estimators=300, random_state=156)
scores = cross_val_score(rcf, X_features, y_target, scoring='accuracy', cv=3)

print('CV=3 인 경우의 개별 Fold세트별 정확도:', scores)
print('평균 정확도:{0:.4f}'.format(np.mean(scores)))

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 전체 23개 피처를 모두 표준화 (PCA를 전체 피처에 적용하기 위함)
scaler = StandardScaler()
df_scaled = scaler.fit_transform(X_features)

# 6개의 주성분으로 차원 축소 (23 -> 6, 약 1/4 수준으로 압축)
pca = PCA(n_components=6)
df_pca = pca.fit_transform(df_scaled)
# 동일한 랜덤 포레스트 모델로 교차 검증 수행
scores_pca = cross_val_score(rcf, df_pca, y_target, scoring='accuracy', cv=3)

print('CV=3 인 경우의 PCA 변환된 개별 Fold세트별 정확도:', scores_pca)
print('PCA 변환 데이터 셋 평균 정확도:{0:.4f}'.format(np.mean(scores_pca)))
# 결과 해석: 피처를 1/4 수준으로 줄였음에도 원본 대비 정확도 하락 폭이 매우 작음
# 데이터셋에 존재하던 다중공선성이 PCA로 효과적으로 압축되었음을 확인할 수 있음

### 6.2 핵심 정리
- PCA는 데이터의 분산을 최대한 보존하는 새로운 직교 축(주성분)을 찾아 차원을 축소하는 비지도 학습 기법이다.
- 공분산 행렬의 고유값 분해를 통해 주성분이 도출되며, 큰 고유값을 가진 고유벡터부터 PC1, PC2 순으로 채택된다.
- PCA 적용 전에는 반드시 표준화가 필요하다. 단, StandardScaler는 데이터를 평균 0, 분산 1로 조정할 뿐 분포의 형태 자체를 정규분포로 바꾸지는 않는다.
- `explained_variance_ratio_`로 각 주성분이 설명하는 분산 비율을 확인할 수 있으며, 이를 기반으로 적절한 컴포넌트 수를 결정한다.
- 상관관계가 높은 피처가 많은 데이터일수록 PCA의 차원 축소 효과가 크다. Credit Card 사례처럼 다중공선성을 가진 변수군에 특히 효과적이다.